# Лабораторная работа 14: сегментация изображения методом K‑Means

In [7]:
import numpy as np
from skimage import io
from skimage import img_as_float
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error
import math

## 1. Загрузите картинку parrots.jpg. Преобразуйте изображение, приведя все значения в интервал от 0 до 1 (функция img_as_float).

In [11]:
img = io.imread('parrots.jpg')
img = img_as_float(img)

## 2. Создайте матрицу объекты-признаки: характеризуйте каждый пиксель тремя координатами — значениями интенсивности в пространстве RGB.

In [12]:
pxls = img.reshape(-1, 3)

## 3. Реализуйте PSNR (метрика качества) и функцию, которая для заданного числа кластеров запускает K‑Means (init='k‑means++', random_state=241) и заполняет пиксели каждого кластера средним либо медианным цветом.

## 4. Измерьте качество получившейся сегментации с помощью PSNR.

In [15]:
def psnr(original, comp):
    mse = mean_squared_error(original.flatten(), comp.flatten())
    max_pixel = 1.0
    psnr_value = 20 * math.log10(max_pixel / math.sqrt(mse))
    return psnr_value

def compress_with_kmeans(n_clstrs, pxls, original_shape, method='mean'):
    kmeans = KMeans(n_clusters=n_clstrs, init='k-means++', random_state=241, n_init=10)
    lbls = kmeans.fit_predict(pxls)
    
    comp = np.zeros_like(pxls)
    for i in range(n_clstrs):
        clstr_mask = (lbls == i)
        if method == 'mean':
            fill_value = pxls[clstr_mask].mean(axis=0)
        elif method == 'median':
            fill_value = np.median(pxls[clstr_mask], axis=0)
        comp[clstr_mask] = fill_value
    
    return comp.reshape(original_shape)

## 5. Найдите минимальное количество кластеров (не более 20), при котором значение PSNR выше 20 (хотя бы для одного из способов заполнения). Это число и будет ответом.

In [18]:
original_shape = img.shape
res = {}

for n_clstrs in range(1, 21):
    comp_mean = compress_with_kmeans(n_clstrs, pxls, original_shape, method='mean')
    psnr_mean = psnr(img, comp_mean)
    
    comp_median = compress_with_kmeans(n_clstrs, pxls, original_shape, method='median')
    psnr_median = psnr(img, comp_median)
    
    res[n_clstrs] = (psnr_mean, psnr_median)
    
    print(f" n_clstrs: {n_clstrs}  Mean PSNR: {psnr_mean:.2f}, Median PSNR: {psnr_median:.2f}")
    
    if psnr_mean > 20 or psnr_median > 20:
        ans = n_clstrs
        break

with open('1.txt', 'w') as f:
    f.write(f'{ans}')

 n_clstrs: 1  Mean PSNR: 9.82, Median PSNR: 9.43
 n_clstrs: 2  Mean PSNR: 12.08, Median PSNR: 11.65
 n_clstrs: 3  Mean PSNR: 13.15, Median PSNR: 12.79
 n_clstrs: 4  Mean PSNR: 14.37, Median PSNR: 14.01
 n_clstrs: 5  Mean PSNR: 15.53, Median PSNR: 15.18
 n_clstrs: 6  Mean PSNR: 16.54, Median PSNR: 16.05
 n_clstrs: 7  Mean PSNR: 17.64, Median PSNR: 17.34
 n_clstrs: 8  Mean PSNR: 18.44, Median PSNR: 18.14
 n_clstrs: 9  Mean PSNR: 19.11, Median PSNR: 18.82
 n_clstrs: 10  Mean PSNR: 19.64, Median PSNR: 19.42
 n_clstrs: 11  Mean PSNR: 20.13, Median PSNR: 19.85
